# What are embeddings?

OpenAI’s text embeddings measure the relatedness of text strings. Embeddings are commonly used for:

- Search (where results are ranked by relevance to a query string)
- Clustering (where text strings are grouped by similarity)
- Recommendations (where items with related text strings are recommended)
- Anomaly detection (where outliers with little relatedness are identified)
- Diversity measurement (where similarity distributions are analyzed)
- Classification (where text strings are classified by their most similar label)

An embedding is a vector (list) of floating point numbers. The distance between two vectors measures their relatedness. Small distances suggest high relatedness and large distances suggest low relatedness.



In [9]:
# Get embeddings
from openai import OpenAI
import json
client = OpenAI()

response = client.embeddings.create(
    input="Your text string goes here",
    model="text-embedding-3-small"
)

data = response.model_dump()
data['data'][0]['embedding'] = f"[...{len(response.data[0].embedding)} floats...]"
print(json.dumps(data, indent=2))

{
  "data": [
    {
      "embedding": "[...1536 floats...]",
      "index": 0,
      "object": "embedding"
    }
  ],
  "model": "text-embedding-3-small",
  "object": "list",
  "usage": {
    "prompt_tokens": 5,
    "total_tokens": 5
  }
}


# Use cases

[Amazone fine-food reviews dataset](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)

In [16]:
import pandas as pd

df = pd.read_csv('fine_food_reviews_1k.csv')
df = df[["Time", "ProductId", "UserId", "Score", "Summary", "Text"]]
df = df.dropna()
df["combined"] = (
    "Title: " + df.Summary.str.strip() + "; Content: " + df.Text.str.strip()
)

df.head()

,Time,ProductId,UserId,Score,Summary,Text,combined
0,1303862400,B001E4KFG0,A3SGXH7AUHU8GW,5,Good Quality Dog Food,I have bought several of the Vitality canned d...,Title: Good Quality Dog Food; Content: I have ...
1,1346976000,B00813GRG4,A1D87F6ZCVE5NK,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Title: Not as Advertised; Content: Product arr...
2,1219017600,B000LQOCH0,ABXLMWJIXXAIN,4,"""Delight"" says it all",This is a confection that has been around a fe...,"Title: ""Delight"" says it all; Content: This is..."
3,1307923200,B000UA0QIQ,A395BORC6FGVXV,2,Cough Medicine,If you are looking for the secret ingredient i...,Title: Cough Medicine; Content: If you are loo...
4,1350777600,B006K2ZZ7K,A1UQRSCLF8GW1T,5,Great taffy,Great taffy at a great price. There was a wid...,Title: Great taffy; Content: Great taffy at a ...


# 调用API获取向量embeddings

Get_embeddings_from_dataset.ipynb

In [24]:
from openai import OpenAI
import httpx

client = OpenAI(http_client=httpx.Client(verify=False))

In [ ]:
import os

OUTPUT_PATH = 'output'
EMBEDDED_FILE = 'embedded_1k_reviews.csv'

os.mkdir(OUTPUT_PATH) if not os.path.exists(OUTPUT_PATH) else None

embedding_model = "text-embedding-3-small"
embedding_encoding = "cl100k_base"
max_tokens = 8000  # the maximum for text-embedding-3-small is 8191

def get_embedding(text, model=embedding_model):
    text = text.replace("\n", " ")
    return client.embeddings.create(input = [text], model=model).data[0].embedding

df['ada_embedding'] = df.combined.apply(lambda x: get_embedding(x, model=embedding_model))
df.to_csv(os.path.join(OUTPUT_PATH, EMBEDDED_FILE), index=False)


# 减小embedding维度

In [ ]:
import numpy as np

def normalize_l2(x):
    x = np.array(x)
    if x.ndim == 1:
        norm = np.linalg.norm(x)
        if norm == 0:
            return x
        return x / norm
    else:
        norm = np.linalg.norm(x, 2, axis=1, keepdims=True)
        return np.where(norm == 0, x, x / norm)


response = client.embeddings.create(
    model="text-embedding-3-small", input="Testing 123", encoding_format="float"
)

cut_dim = response.data[0].embedding[:256]
norm_dim = normalize_l2(cut_dim)

print(len(norm_dim))

256


# 使用基于embeddings搜索的问答

Question_answering_using_embeddings.ipynb

## 完整流程

- Prepare search data (once per document)
  - Collect: We'll download a few hundred Wikipedia articles about the 2022 Olympics
  - Chunk: Documents are split into short, mostly self-contained sections to be embedded
  - Embed: Each section is embedded with the OpenAI API
  - Store: Embeddings are saved (for large datasets, use a vector database)
- Search (once per query)
  - Given a user question, generate an embedding for the query from the OpenAI API
  - Using the embeddings, rank the text sections by relevance to the query
- Ask (once per query)
  - Insert the question and the most relevant sections into a message to GPT
  - Return GPT's answer
